# 07: El viaje de un dato: de su origen hasta una decisión de negocio


## Ejemplo concreto: el pipeline de una tienda online


La tienda "PixelShop" vende electrónica online donde cada noche necesita un reporte de ventas por producto para que el equipo de compras sepa qué reponer. 

Veamos el viaje de los datos:

1.  **ORIGEN:** la app de la tienda guarda cada pedido en un archivo JSON diario (pedidos_2026-01-15.json).
2.  **EXTRAER:** nuestro script Python lee ese archivo JSON.
3.  **TRANSFORMAR:** limpiamos datos (quitar pedidos cancelados, convertir moneda, validar importes), enriquecemos (calcular importe total = precio × cantidad) y agregamos (sumar por producto).
4.  **CARGAR:** escribimos el resultado en un archivo Parquet organizado por fecha.
5.  **CONSUMIR:** el equipo de compras consulta un dashboard que lee esos Parquets y muestra qué reponer.


In [1]:
import json, csv
from datetime import date, timedelta
from collections import defaultdict

### EXTRACT: leer datos del origen (simulamos el JSON de pedidos del día)

In [2]:
pedidos_raw = [
    {"id": 1, "producto": "Mouse", "precio": 25.0, "cantidad": 2, "estado": "completado"},
    {"id": 2, "producto": "Teclado", "precio": 60.0, "cantidad": 1, "estado": "completado"},
    {"id": 3, "producto": "Mouse", "precio": 25.0, "cantidad": 1, "estado": "cancelado"},
    {"id": 4, "producto": "Monitor", "precio": 350.0, "cantidad": 1, "estado": "completado"},
    {"id": 5, "producto": "Teclado", "precio": 60.0, "cantidad": 3, "estado": "completado"},
    {"id": 6, "producto": "Webcam", "precio": 80.0, "cantidad": 1, "estado": "completado"},
]
print(f"EXTRACT: {len(pedidos_raw)} pedidos leídos")

EXTRACT: 6 pedidos leídos


# TRANSFORM: limpiar + enriquecer + agregar


In [3]:
# 1. Filtrar: solo pedidos completados
pedidos_validos = [p 
                   for p in pedidos_raw 
                   if p['estado'] == 'completado']

print(f"TRANSFORM: {len(pedidos_validos)} pedidos válidos (sin cancelados)")


TRANSFORM: 5 pedidos válidos (sin cancelados)


In [7]:
# 2. Enriquecer: Calcular importe total

for p in pedidos_validos:
    p['importe_total'] = p['precio'] * p['cantidad']

In [15]:
# 3. Agregar: sumar por producto
ventas_por_producto = defaultdict(lambda: {'unidades': 0, 'importe': 0})

for p in pedidos_validos:
    ventas_por_producto[p['producto']]['unidades'] += p['cantidad']
    ventas_por_producto[p['producto']]['importe'] += p['importe_total']

ventas_por_producto


defaultdict(<function __main__.<lambda>()>,
            {'Mouse': {'unidades': 2, 'importe': 50.0},
             'Teclado': {'unidades': 4, 'importe': 240.0},
             'Monitor': {'unidades': 1, 'importe': 350.0},
             'Webcam': {'unidades': 1, 'importe': 80.0}})

### LOAD: guardar resultado


In [16]:
hoy = date.today().isoformat()

with open(f"reporte_ventas_{hoy}.csv", 'w', newline="") as f:
    writer = csv.writer(f)
    writer.writerow(['producto', 'unidades_vendidas', 'importe_total', "fecha"])

    for prod, datos in sorted(ventas_por_producto.items(), key=lambda x: -x[1]['importe']):
        writer.writerow([prod, datos['unidades'], datos['importe'], hoy])


In [17]:
print(f"LOAD: reporte guardado como reporte_ventas_{hoy}.csv")
print("\n=== RESULTADO (lo que ve el equipo de compras) ===")
for prod, datos in sorted(ventas_por_producto.items(), key=lambda x: -x[1]["importe"]):
    print(f"  {prod:10} | {datos['unidades']} uds | {datos['importe']:>8.2f}€")


LOAD: reporte guardado como reporte_ventas_2026-10-10.csv

=== RESULTADO (lo que ve el equipo de compras) ===
  Monitor    | 1 uds |   350.00€
  Teclado    | 4 uds |   240.00€
  Webcam     | 1 uds |    80.00€
  Mouse      | 2 uds |    50.00€


In [ ]:
7